# Part 5a: The Vitis Unified backend

The previous parts of this tutorial stop at the HLS IP: a piece of RTL that still has to be integrated into a full system by hand. The **VitisUnified** backend of hls4ml goes further. It wraps the hls4ml kernel with an AXI interface, links it into a SoC platform (processing system, interconnect, interrupt controller, DMA) with `v++`, and runs place and route, so that the output is a bitstream plus a PYNQ driver ready to run on a board.

Part 5 walks through the full flow:

| Notebook | What it does |
|----------|--------------|
| **5a** (this one) | Converts a small U-Net with both the `VitisUnified` and the `Vitis` backends and checks that the software predictions are identical |
| 5b | Runs **C simulation** (`csim`) and compares it with the software prediction. If this passes, the generated C++ is correct |
| 5c | Runs **RTL co-simulation** (`cosim`): the C++ is synthesized to RTL and simulated |
| 5d | Enables the **FIFO depth optimization** flow, which profiles the FIFOs during co-simulation and writes `fifo_depths.json` |
| 5e | Generates the **bitfile**: synthesis, system link, place and route. Writes `system.bit`, `system.hwh` and the PYNQ driver |
| 5f | A guide to building your own SoC platform in Vivado and adding a new board to the backend |

## Prerequisites

- Vitis and Vivado **2023.2** (tested). The `kv260` platform also supports 2025.2.
- A supported board: **zcu102** or **kv260**. The backend ships a platform for both, so you do not need to build one yourself.
- hls4ml with the Vitis Unified backend. The backend is not merged into the main branch of hls4ml yet; install it from pull request [#1376](https://github.com/fastmachinelearning/hls4ml/pull/1376):

```bash
pip install git+https://github.com/fastmachinelearning/hls4ml.git@refs/pull/1376/head
```

In this notebook we only use the C++ bridge (`hls_model.predict`), so no Vitis run is needed yet, but the backend still needs `XILINX_VITIS` to locate the platform.

## Settings

Set the target board here. The backend picks the platform (`.xpfm` or `.xsa`) from its `supported_boards.json` using the board and the AXI mode, so nothing else needs to be configured. To add another board, see `5_vu_f_platform_setup.md`.

| Variable | Meaning |
|----------|---------|
| `BOARD` | `'zcu102'` or `'kv260'` |
| `AXI_MODE` | `'axi_master'` (default) or `'axi_stream'`; see the *AXI interface modes* page of the hls4ml docs |

The Vitis tools are located through the `XILINX_VITIS` environment variable, as in the other parts of this tutorial.

In [ ]:
import os

os.environ['KERAS_BACKEND'] = 'tensorflow'

import numpy as np
import sys

sys.path.append('../..')
from models import create_simple_unet
import hls4ml

os.environ['PATH'] = os.environ['XILINX_VITIS'] + '/bin:' + os.environ['PATH']

BOARD = 'zcu102'  # or 'kv260'
AXI_MODE = 'axi_master'  # or 'axi_stream'

DATA_DIR = '../../data/vitis_unified'
os.makedirs(DATA_DIR, exist_ok=True)
np.random.seed(0)

## Create the model and some test data

We use a tiny U-Net with a single skip connection, defined in `models.py`. The weights are left untrained on purpose: the goal of this part is to check that every stage of the accelerator flow reproduces the same numbers, not to reach a good accuracy. A `Concatenate` skip connection is a good stress test for the streaming interface, because it forces two branches to be merged again.

In [ ]:
X = np.random.rand(10, 4, 4, 1).astype(np.float32)
np.save(f'{DATA_DIR}/X_part5a.npy', X)
print(f'Input shape: {X.shape}')

model = create_simple_unet()
model.summary()

## Convert with both backends

We convert the same model twice: once with the `VitisUnified` backend and once with the plain `Vitis` backend as a reference. Both use `io_stream`, which is required for the CNN layers.

The `VitisUnified` backend takes the same arguments as the `Vitis` backend, plus:

| Argument | Meaning |
|----------|---------|
| `board` | the target board; the FPGA part and the platform are looked up from it |
| `axi_mode` | `'axi_master'` or `'axi_stream'`, the interface between the PS and the kernel |
| `input_type`, `output_type` | the data type on the AXI interface (`'float'` here, so the driver can send `float32` NumPy arrays directly) |

In [ ]:
config = hls4ml.utils.config_from_keras_model(model, granularity='name', backend='Vitis')

hls_model_unified = hls4ml.converters.convert_from_keras_model(
    model,
    hls_config=config,
    backend='VitisUnified',
    io_type='io_stream',
    output_dir=os.path.abspath(f'../../hls4ml_prjs/hls4ml_prj_part5a_unified_{AXI_MODE}'),
    board=BOARD,
    axi_mode=AXI_MODE,
    input_type='float',
    output_type='float',
    clock_period=10,
)
hls_model_unified.compile()

In [ ]:
hls_model_vitis = hls4ml.converters.convert_from_keras_model(
    model,
    hls_config=config,
    backend='Vitis',
    io_type='io_stream',
    output_dir='../../hls4ml_prjs/hls4ml_prj_part5a_vitis',
    part='xczu9eg-ffvb1156-2-e',
    clock_period=10,
)
hls_model_vitis.compile()

## Predict and compare

The AXI wrapper added by the `VitisUnified` backend must not change the arithmetic: the fixed-point results of the two backends should be bit-for-bit identical.

In [ ]:
y_unified = hls_model_unified.predict(X)
y_vitis = hls_model_vitis.predict(X)

print(f'VitisUnified output shape: {np.array(y_unified).shape}')
print(f'Vitis        output shape: {np.array(y_vitis).shape}')

assert np.array_equal(y_unified, y_vitis), 'The results from VitisUnified and Vitis are NOT equal!'
print('Backend prediction comparison passed: both are equal')

Continue with `5_vu_b_csim.ipynb` to check that the generated C++ behaves the same when compiled by the Vitis HLS compiler.